# Minimal example

The smallest useful network: three stations on a medium-voltage cable line, a
current source at one end and an earth fault at the other. The notebook builds
it, solves it, reads the results as tables, and checks four of them against
what Kirchhoff's laws and the closed forms of a cable line demand.

For a step-by-step introduction in which every number is also computed by
hand, see the tutorial
[Your first fault calculation](../../getting-started/first-calculation/).

In [ ]:
import polars as pl

import groundinsight as gi

print("groundinsight", gi.__version__)

## 1. The network

Three stations `bus1`, `bus2`, `bus3` in a line, joined by two 1 km sections
of a cable whose shield is earthed at every station. Each station has an
electrode of 1 Ω. 100 A at 50 Hz enter at `bus1`, the fault is at `bus3`.
The values are illustrative round numbers, chosen so that the closed forms
below stay short.

In [ ]:
Z_SELF = complex(0.25, 0.6)    # shield-earth loop, Ohm/km at 50 Hz
Z_MUTUAL = complex(0.0, 0.6)   # phase-shield coupling, Ohm/km at 50 Hz
Z_BUS = 1.0                    # electrode of every station, Ohm
I_SOURCE = 100.0               # A, RMS

station = gi.BusType(
    name="Station", system_type="Substation", voltage_level=20.0,
    impedance_formula="1.0",
)
cable = gi.BranchType(
    name="MVCable", grounding_conductor=True,
    self_impedance_formula="(0.25 + j * 0.6) * l",
    mutual_impedance_formula="(0.0 + j * 0.6) * l",
)

net = gi.create_network(name="minimal", frequencies=[50.0])
for i in range(1, 4):
    gi.create_bus(name=f"bus{i}", type=station, network=net)
gi.create_branch(name="b12", type=cable, from_bus="bus1", to_bus="bus2", length=1.0, network=net)
gi.create_branch(name="b23", type=cable, from_bus="bus2", to_bus="bus3", length=1.0, network=net)
gi.create_source(name="src", bus="bus1", values={50.0: I_SOURCE}, network=net)
gi.create_fault(name="fault", bus="bus3", network=net)
print(f"{len(net.buses)} buses, {len(net.branches)} branches")

## 2. Solve and read the results

`run_fault` returns the result and keeps it on the network. The `res_*`
methods turn it into Polars tables: `res_buses` per bus and frequency (plus an
RMS row, marked by `is_rms`), `res_branches` per branch, and `res_fault` the
key figures of the fault.

In [ ]:
result = gi.run_fault(net, fault="fault")

per_frequency = ~pl.col("is_rms")
print(net.res_buses(fault="fault").filter(per_frequency)
      .select("bus_name", "EPR_V", "EPR_degree", "I_bus_A", "I_bus_degree"))
print(net.res_branches(fault="fault").filter(per_frequency)
      .select("branch_name", "I_branch_A", "I_branch_degree"))
net.res_fault("fault").filter(per_frequency).select(
    "U_E_V", "Z_E_Ohm", "Z_E_degree", "I_E_A", "r_current", "r_epr"
)

## 3. Four checks

**(a) Kirchhoff over the whole network.** Whatever enters the soil at one
electrode leaves it at another, so the electrode currents of all buses add up
to zero — as complex numbers, not as magnitudes.

**(b) Symmetry.** The line is uniform and the electrodes are equal. The
coupling of the phase current adds up to a current $(1 - k)\,I$ that enters
the earthing network at `bus1` and leaves it at `bus3`, so the potentials are
antisymmetric: `bus2` stays at 0 V, and `bus1` and `bus3` rise to the same
magnitude with opposite sign.

**(c) The EPR ratio** of a line with one cable type is
$r = |1 - Z_m/Z_s|$, whatever the electrodes.

**(d) The reduction factor of EN 50522**, $|I_E|/|I|$: with `bus2` at 0 V the
fault bus sees its own electrode in parallel with one section, and
$|I_E|/|I| = |Z_s - Z_m| / |Z_B + Z_s|$ for one section.

In [ ]:
electrode = {b.name: complex(b.ia_freq[50.0].real, b.ia_freq[50.0].imag) for b in result.buses}
epr = {b.name: complex(b.uepr_freq[50.0].real, b.uepr_freq[50.0].imag) for b in result.buses}
factor = result.reduction_factor

print(f"(a) |sum of the electrode currents| = {abs(sum(electrode.values())):.2e} A")
print(f"(b) EPR(bus2) = {abs(epr['bus2']):.2e} V,  EPR(bus1) + EPR(bus3) = "
      f"{abs(epr['bus1'] + epr['bus3']):.2e} V,  |EPR(bus3)| = {abs(epr['bus3']):.4f} V")
r_closed = abs(1 - Z_MUTUAL / Z_SELF)
print(f"(c) EPR ratio: model {factor.value[50.0]:.6f}, closed form {r_closed:.6f}")
r_i_closed = abs(Z_SELF - Z_MUTUAL) / abs(Z_BUS + Z_SELF)
print(f"(d) r = |I_E|/|I|: model {factor.value_current[50.0]:.6f}, closed form {r_i_closed:.6f}")

assert abs(sum(electrode.values())) < 1e-9 * I_SOURCE
assert abs(epr["bus2"]) < 1e-9 and abs(epr["bus1"] + epr["bus3"]) < 1e-9
assert abs(factor.value[50.0] - r_closed) < 1e-12
assert abs(factor.value_current[50.0] - r_i_closed) < 1e-12

## What the model assumes

- One linear solve per frequency in the phasor domain; the potentials are
  earth potential rises against remote earth.
- Every station is a lumped electrode; the soil between the stations does not
  couple them.
- The fault current returns through the soil and through the cable shields.
  The phase current induces a voltage along each shield, which enters the
  solve as a Norton current; on a line the phase current is the source
  current itself.
- No fault resistance: the fault bonds the phase conductor to the earthing
  system of `bus3`.

[Model and assumptions](../../explanation/theory/) lists the assumptions in
full, and [Conventions and units](../../reference/conventions/) explains the
angles in the tables above.